# pyspi

pyspi computes **statistics of pairwise interactions** (SPIs) between the processes of a multivariate time series. Each SPI is one answer to "how are these two processes related?" — a correlation, a spectral coherence, a transfer entropy, a causal score. For `M` processes each returns an `M x M` matrix, and pyspi stacks them so hundreds of notions of "related" can be compared side by side.

## 1. Data

In [ ]:
import numpy as np
from pyspi.calculator import Calculator, bundled_configs
from pyspi.data import load_dataset, available_datasets

available_datasets()

`Data` holds an `M x T` array, z-scored per process by default. Your own data goes in the same way — `Calculator(dataset=my_array)` accepts a NumPy array, and `dim_order` controls whether rows are processes or observations.

In [ ]:
data = load_dataset("cml")   # coupled map lattice: 10 processes, 500 observations
print(data.n_processes, "processes x", data.n_observations, "observations")

## 2. Compute

`config` selects the SPI set. `fabfour` is the smallest (4 SPIs, one per family); `full` is all 322. See `bundled_configs()`.

In [ ]:
calc = Calculator(dataset=data, 
                  config="benchmarked_p90", 
                  verbose=True)
calc.compute(progress=False)

calc.table

## 3. Reading the results

`calc` itself shows what happened. `calc.table` is wide: rows are processes, columns a `(spi, process)` MultiIndex, shape `M x (n_spis * M)`.

In [ ]:
calc.summary()

In [ ]:
print(calc.table.shape, calc.table.columns.names)
calc.table.iloc[:3, :6].round(3)

### One SPI

Indexing by identifier returns that SPI's `M x M` matrix. Entry `[i, j]` is computed with `i` as **source**, `j` as **target** — irrelevant for a symmetric SPI, the whole point for a directed one. The diagonal is NaN: self-pairs are not computed.

In [ ]:
cov = calc.table["cov_EmpiricalCovariance"]
cov.round(3)

In [ ]:
# a single pair, and the strongest pairs overall
print("proc-0 / proc-3:", round(cov.iloc[0, 3], 4))

upper = cov.where(np.triu(np.ones(cov.shape), k=1).astype(bool))
upper.stack().abs().sort_values(ascending=False).head(5).round(3)

### Direction

The data is z-scored, so covariance *is* Pearson correlation and is symmetric. Directed information is not — the two triangles are separate estimates.

In [ ]:
di = calc.table["di_gaussian_n-5"]
print("0 -> 1:", round(di.iloc[0, 1], 3), " | 1 -> 0:", round(di.iloc[1, 0], 3))
print("di symmetric?", np.allclose(di.values, di.values.T, equal_nan=True),
      "| cov symmetric?", np.allclose(cov.values, cov.values.T, equal_nan=True))

### Long form

`to_frame()` gives one row per `(spi, source, target)` — the shape you want for plotting, `groupby`, or joining against SPI labels.

In [ ]:
long = calc.to_frame()
display(long.head(3))
long.groupby("spi")["value"].agg(["mean", "std", "count"]).round(3)

## 4. Cost

Cost is heavily skewed: a few SPIs dominate, most are effectively free. `summary()` reports the total and the slowest five, which is how to budget a run.

The `benchmarked_p80/p90/p95/p99` configs exploit that skew — they keep the fastest N% by measured cost. To build your own subset by keyword, `filter_spis(["directed", "nonlinear"], output_name="mine")` writes a config you pass as `Calculator(config="mine.yaml")`.

In [ ]:
import time

t0 = time.perf_counter()
sonnet = Calculator(dataset=data, config="sonnet", verbose=False)   # 14 SPIs, one per family
sonnet.compute(progress=False)
print(f"{sonnet.n_spis} SPIs in {time.perf_counter() - t0:.1f}s")

sonnet.summary()["slowest"]

## 5. Failures

A failed SPI still occupies its column, filled with NaN. Check `calc.errors` before reading a table: a NaN column is otherwise indistinguishable from a legitimately undefined statistic.

In [ ]:
sonnet.errors or "no failures"

## 6. Inspecting the calculator

What was run, what it cost, and what is available.

In [ ]:
print("SPIs:", calc.n_spis)
print("first five:", list(calc.spis)[:5])

# every parameter that defines this run -- what checkpoints are bound to
{k: v for k, v in calc.run_spec.items() if k != "spi_identifiers"}

In [ ]:
# labels drive filtering; each SPI carries the traits it actually has
spi = calc.spis["di_gaussian_n-5"]
print(spi.identifier, "->", sorted(spi.labels))

# build your own subset by keyword
from pyspi.utils import filter_spis
# filter_spis(["directed", "nonlinear"], output_name="mine")   # -> mine.yaml

## 6. Saving, scale, CLI

`.npz` round-trips exactly and needs nothing but numpy; `.csv` is a one-way export.

```python
calc.save("results.npz")
from pyspi.calculator import load_table
table = load_table("results.npz")
```

**Many datasets: run one per process** (a cluster array job, or GNU parallel) rather than raising `n_jobs`. SPIs sharing a cache are grouped into one sequential task, so within-dataset speedup is floored by the longest group — measured at 2.3–4.5x whatever `n_jobs` you pass. One dataset per process scales close to linearly and confines a failure to one dataset.

Everything above is available without writing Python:

```bash
python -m pyspi compute --data ts.npy --config fabfour --checkpoint-dir results/
```

`--checkpoint-dir` writes each SPI as it finishes, so an interrupted run resumes instead of restarting.